# Agentic MRO Optimization - Complete Pipeline

This notebook demonstrates the integration of agentic mobility generation with MRO (Mobility Robustness Optimization).

**Pipeline:**
1. **Natural Language → Mobility Simulation**: Generate UE mobility data from text queries
2. **Topology Generation**: LLM-generated cell tower placement
3. **Agentic MRO Optimization**: Intelligent parameter optimization using multi-agent LLM workflow

---

## Part 0: Environment Setup Instructions

### Step 1: Create Virtual Environment

```bash
# Navigate to project root
cd /path/to/maveric

# Create virtual environment
python3 -m venv venv # python v3.10.16 recommened

# Activate (macOS/Linux)
source venv/bin/activate

# Activate (Windows)
venv\Scripts\activate
```

### Step 2: Install Requirements

```bash
pip install -r radp/digital_twin/requirements.txt

pip install -r notebooks/requirements.txt

pip install -r requirements-dev.txt
```

### Step 3: Configure API Keys

**For Agentic Mobility:**
```bash
# Copy example environment file
cp radp/digital_twin/agentic_mobility/.env.example .env

# Edit the .env file and add your API key
# GROQ_API_KEY=your_key_here
```

**For Agentic MRO (when implemented):**
```bash
# Copy example environment file
cp apps/mobility_robustness_optimization/agentic_mro/.env.example apps/mobility_robustness_optimization/agentic_mro/.env

# Edit the .env file and add your API key
```

**Get Groq API Key:** https://console.groq.com/ (free tier available)

### Step 4: Launch Jupyter

```bash
# From project root
jupyter notebook
```

Now you're ready to run this notebook!

---

## Part 1: Introduction & Setup

### What is Agentic MRO?

Agentic MRO combines two powerful AI-driven features:

**1. Agentic Mobility Generation**
- Transform natural language → mobility simulation
- LLM-powered parameter inference
- Automatic geocoding and validation

**2. Agentic MRO Optimization (Coming Soon)**
- Multi-agent LLM workflow for parameter optimization
- Intelligent search instead of brute-force
- Minimizes Radio Link Failures (RLFs) and handover interruptions
- Optimizes Hysteresis and Time-to-Trigger parameters

### What This Notebook Demonstrates:

**Current Implementation:**
1. Natural language → mobility simulation
2. Cell tower topology generation
3. Data preparation for MRO optimization

**Agentic MRO:**
4. Agentic MRO multi-agent optimization
5. Before/after parameter comparison
6. Performance metrics visualization

---

In [15]:
# Setup Python path
import sys
from pathlib import Path

# Add maveric root to path
notebook_dir = Path.cwd()
maveric_root = notebook_dir.parent
if str(maveric_root) not in sys.path:
    sys.path.insert(0, str(maveric_root))

print(f"Notebook directory: {notebook_dir}")
print(f"Maveric root: {maveric_root}")

Notebook directory: /Users/azanzaman/Desktop/maveric1/notebooks
Maveric root: /Users/azanzaman/Desktop/maveric1


In [16]:
# Create output directory
output_dir = Path("data/agentic_data/mro")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory created: {output_dir}")
print(f"  Absolute path: {output_dir.resolve()}")
print(f"  Directory exists: {output_dir.exists()}")

Output directory created: data/agentic_data/mro
  Absolute path: /Users/azanzaman/Desktop/maveric1/notebooks/data/agentic_data/mro
  Directory exists: True


In [17]:
# Import required modules
import json
import pandas as pd

from radp.digital_twin.agentic_mobility.integration import AgenticMobilityIntegration
from radp.digital_twin.agentic_mobility.topology_generator import TopologyGenerator

print("All imports successful!")

All imports successful!


---
## Part 2: Natural Language → Mobility Generation

Generate realistic UE mobility data that will be used for MRO optimization.

The system will:
1. Parse the natural language query
2. Resolve the location to lat/lon bounds
3. Generate RADP-compatible parameters
4. Validate and self-correct if needed
5. Generate mobility simulation data

**Note**: This may take ~5-10 seconds due to LLM API calls and geocoding.

---

In [18]:
# Define natural language query
# For MRO optimization, we want realistic mobility patterns
query = "Generate 50 UEs in urban downtown San Francisco with mix of pedestrians and cars"

print(f"Query: '{query}'")
print("\nProcessing...")
print("This may take 5-10 seconds (LLM + geocoding API calls)")

Query: 'Generate 50 UEs in urban downtown San Francisco with mix of pedestrians and cars'

Processing...
This may take 5-10 seconds (LLM + geocoding API calls)


In [19]:
# Generate mobility data from natural language
df, metadata = AgenticMobilityIntegration.generate_from_natural_language(query)

print(f"\nGenerated {len(df)} mobility points for {metadata['query_intent']['num_ues']} UEs")


Generated 2500 mobility points for 50 UEs


In [20]:
# Display key metadata
query_intent = metadata['query_intent']

print("="*70)
print("GENERATION METADATA")
print("="*70)
print(f"Location: {query_intent['location']}")
print(f"Scenario Type: {query_intent['scenario_type']}")
print(f"Number of UEs: {query_intent['num_ues']}")
print(f"Number of Ticks: {query_intent['num_ticks']}")
print(f"Retry Count: {metadata['retry_count']}")

# Handle ue_distribution structure
ue_dist = query_intent['ue_distribution']
source = ue_dist.get('source', 'unknown')
print(f"\nUE Distribution (Source: {source}):")

# Get distribution values (exclude 'source' key)
for ue_type, percentage in ue_dist.items():
    if ue_type != 'source':
        print(f"  - {ue_type}: {percentage:.1%}")

print("="*70)

GENERATION METADATA
Location: San Francisco
Scenario Type: urban
Number of UEs: 50
Number of Ticks: 50
Retry Count: 0

UE Distribution (Source: predicted):
  - stationary: 20.0%
  - pedestrian: 35.0%
  - cyclist: 10.0%
  - car: 35.0%


In [21]:
# Display DataFrame preview
print("\nDataFrame Preview:")
display(df.head(10))

print(f"\nDataFrame Info:")
print(f"  - Total rows: {len(df)}")
print(f"  - Columns: {list(df.columns)}")
print(f"  - Unique UEs: {df['mock_ue_id'].nunique()}")
print(f"  - Ticks: {df['tick'].min()} to {df['tick'].max()}")
print(f"  - Lat range: [{df['lat'].min():.6f}, {df['lat'].max():.6f}]")
print(f"  - Lon range: [{df['lon'].min():.6f}, {df['lon'].max():.6f}]")


DataFrame Preview:


,mock_ue_id,lon,lat,tick
0,0,-122.360943,37.824207,0
1,1,-122.384754,37.783952,0
2,2,-122.434210,37.761122,0
3,3,-122.354035,37.770455,0
4,4,-122.375743,37.795073,0
5,5,-122.382795,37.758817,0
6,6,-122.413293,37.820064,0
7,7,-122.433484,37.811226,0
8,8,-122.453530,37.807708,0
9,9,-122.434084,37.763622,0



DataFrame Info:
  - Total rows: 2500
  - Columns: ['mock_ue_id', 'lon', 'lat', 'tick']
  - Unique UEs: 50
  - Ticks: 0 to 49
  - Lat range: [37.742962, 37.832925]
  - Lon range: [-122.464023, -122.353502]


In [22]:
# Save mobility data
csv_path = output_dir / "ue_mobility_data.csv"
metadata_path = output_dir / "ue_mobility_metadata.json"

df.to_csv(csv_path, index=False)
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved mobility data to: {csv_path}")
print(f"Saved metadata to: {metadata_path}")

Saved mobility data to: data/agentic_data/mro/ue_mobility_data.csv
Saved metadata to: data/agentic_data/mro/ue_mobility_metadata.json


---
## Part 3: Topology Generation

Generate cell tower topology using LLM-based intelligent placement.

The `TopologyGenerator` uses an LLM to:
- Understand the area type (urban/suburban/rural)
- Calculate optimal number of cell sites
- Place towers intelligently within spatial bounds
- Generate realistic cell configurations

This topology will be used by the MRO optimizer to calculate handover parameters.

---

In [23]:
# Extract location data
location_data = metadata.get('location_data', {})
query_intent = metadata['query_intent']

# Generate topology using LLM
print("Generating cell tower topology using LLM...")
print("This may take a few seconds...\n")

topology_df = TopologyGenerator.generate_from_llm(
    raw_query=query_intent.get('raw_query', query),
    area_type=location_data.get("area_type", "suburban"),
    num_ues=query_intent.get("num_ues", 50),
    min_lat=location_data.get("min_lat", df["lat"].min()),
    max_lat=location_data.get("max_lat", df["lat"].max()),
    min_lon=location_data.get("min_lon", df["lon"].min()),
    max_lon=location_data.get("max_lon", df["lon"].max()),
)

print(f"Generated {len(topology_df)} cell sectors")

Generating cell tower topology using LLM...
This may take a few seconds...


[Topology Generation - LLM Output]
  Reasoning: The raw query does not specify an explicit tower count, so the number of sites is derived from the suburban area size (≈100 km²). Suburban deployments typically use 1 site per 2‑5 km²; choosing the higher end of the density (≈1 site per 4.5 km²) yields about 22 sites, which also accounts for the modest UE count (50) – a relatively low load that does not require the maximum density. A mid‑band carrier (2100 MHz) balances coverage and capacity for suburban environments and supports both pedestrian and vehicular users. GA‑based azimuth optimization works best with directional (sectored) antennas, allowing each site to steer its beam for optimal SINR. The area is fairly uniform with no explicit hotspot or boundary constraints, so a uniform grid placement is the most straightforward and effective strategy.
  num_cells: 22
  frequency: 2100 MHz
  azimuth_strategy: sect

In [24]:
# Display topology preview
print("\nTopology Preview:")
display(topology_df.head(10))

# Count unique cell sites
unique_locations = topology_df.groupby(["cell_lat", "cell_lon"]).size()
print(f"\nGenerated {len(unique_locations)} unique cell sites")


Topology Preview:


,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,37.760714,-122.443939,cell_1,0,2100
1,37.760714,-122.443939,cell_2,120,2100
2,37.760714,-122.443939,cell_3,240,2100
3,37.760714,-122.402779,cell_4,0,2100
4,37.760714,-122.402779,cell_5,120,2100
5,37.760714,-122.402779,cell_6,240,2100
6,37.760714,-122.361620,cell_7,0,2100
7,37.760714,-122.361620,cell_8,120,2100
8,37.760714,-122.361620,cell_9,240,2100
9,37.796359,-122.423359,cell_10,0,2100



Generated 7 unique cell sites


In [25]:
# Save topology
topology_path = output_dir / "cell_topology.csv"
topology_df.to_csv(topology_path, index=False)

# Save generation parameters
params_info = {
    "query": query,
    "location_data": location_data,
    "query_intent": query_intent,
    "metadata": metadata,
}
params_path = output_dir / "generation_params.json"
with open(params_path, 'w') as f:
    json.dump(params_info, f, indent=2)

print(f"Saved topology to: {topology_path}")
print(f"Saved generation parameters to: {params_path}")

Saved topology to: data/agentic_data/mro/cell_topology.csv
Saved generation parameters to: data/agentic_data/mro/generation_params.json


In [26]:
# Validate spatial boundaries
print("\nBoundary Validation:")
print("="*70)
print(f"  UE Lat range: [{df['lat'].min():.6f}, {df['lat'].max():.6f}]")
print(f"  UE Lon range: [{df['lon'].min():.6f}, {df['lon'].max():.6f}]")
print(f"  Cell Lat range: [{topology_df['cell_lat'].min():.6f}, {topology_df['cell_lat'].max():.6f}]")
print(f"  Cell Lon range: [{topology_df['cell_lon'].min():.6f}, {topology_df['cell_lon'].max():.6f}]")

# Check towers within UE bounds
towers_in_bounds = topology_df[
    (topology_df['cell_lat'] >= df['lat'].min()) & 
    (topology_df['cell_lat'] <= df['lat'].max()) &
    (topology_df['cell_lon'] >= df['lon'].min()) & 
    (topology_df['cell_lon'] <= df['lon'].max())
]
print(f"\n  Cell towers within UE bounds: {len(towers_in_bounds)}/{len(topology_df)}")
print("="*70)


Boundary Validation:
  UE Lat range: [37.742962, 37.832925]
  UE Lon range: [-122.464023, -122.353502]
  Cell Lat range: [37.760714, 37.832004]
  Cell Lon range: [-122.443939, -122.361620]

  Cell towers within UE bounds: 21/21


---
## Data Summary for MRO Optimization

The generated mobility and topology data is now ready for MRO optimization.

> for detailed visualization and more, please checkout `notebooks/agentic_mobility_model.ipynb`

Below is a summary of the data that will be used:

---

In [27]:
# Print UE DataFrame summary
print("="*70)
print("UE MOBILITY DATA SUMMARY")
print("="*70)
print(f"Total UEs: {df['mock_ue_id'].nunique()}")
print(f"Total Ticks: {df['tick'].max() + 1}")
print(f"Total Mobility Points: {len(df)}")
print(f"\nSpatial Coverage:")
print(f"  Latitude: {df['lat'].min():.6f} to {df['lat'].max():.6f}")
print(f"  Longitude: {df['lon'].min():.6f} to {df['lon'].max():.6f}")
print(f"\nColumns: {list(df.columns)}")
print("\nSample data:")
display(df.head())

print("\n" + "="*70)
print("CELL TOPOLOGY SUMMARY")
print("="*70)
print(f"Total Cell Sectors: {len(topology_df)}")
print(f"Unique Cell Sites: {len(unique_locations)}")
print(f"\nSpatial Coverage:")
print(f"  Latitude: {topology_df['cell_lat'].min():.6f} to {topology_df['cell_lat'].max():.6f}")
print(f"  Longitude: {topology_df['cell_lon'].min():.6f} to {topology_df['cell_lon'].max():.6f}")
print(f"\nColumns: {list(topology_df.columns)}")
print("\nSample data:")
display(topology_df.head())

print("\n" + "="*70)
print("DATA READY FOR MRO OPTIMIZATION")
print("="*70)
print(f"Mobility CSV: {csv_path}")
print(f"Topology CSV: {topology_path}")
print(f"Metadata JSON: {metadata_path}")
print("="*70)

UE MOBILITY DATA SUMMARY
Total UEs: 50
Total Ticks: 50
Total Mobility Points: 2500

Spatial Coverage:
  Latitude: 37.742962 to 37.832925
  Longitude: -122.464023 to -122.353502

Columns: ['mock_ue_id', 'lon', 'lat', 'tick']

Sample data:


,mock_ue_id,lon,lat,tick
0,0,-122.360943,37.824207,0
1,1,-122.384754,37.783952,0
2,2,-122.434210,37.761122,0
3,3,-122.354035,37.770455,0
4,4,-122.375743,37.795073,0



CELL TOPOLOGY SUMMARY
Total Cell Sectors: 21
Unique Cell Sites: 7

Spatial Coverage:
  Latitude: 37.760714 to 37.832004
  Longitude: -122.443939 to -122.361620

Columns: ['cell_lat', 'cell_lon', 'cell_id', 'cell_az_deg', 'cell_carrier_freq_mhz']

Sample data:


,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,37.760714,-122.443939,cell_1,0,2100
1,37.760714,-122.443939,cell_2,120,2100
2,37.760714,-122.443939,cell_3,240,2100
3,37.760714,-122.402779,cell_4,0,2100
4,37.760714,-122.402779,cell_5,120,2100



DATA READY FOR MRO OPTIMIZATION
Mobility CSV: data/agentic_data/mro/ue_mobility_data.csv
Topology CSV: data/agentic_data/mro/cell_topology.csv
Metadata JSON: data/agentic_data/mro/ue_mobility_metadata.json


---
## Agentic MRO Optimization (Coming Soon)

**Multi-Agent LLM Workflow for Intelligent Parameter Optimization**

The agentic MRO feature uses a multi-agent LLM system to optimize MRO parameters:

1. **Analyzer Agent**: Analyzes network simulation data (signal quality, mobility patterns, handover risks)
2. **Strategy Agent**: Recommends optimal parameter ranges (Hysteresis, Time-to-Trigger)
3. **Coordinator Agent**: Iteratively suggests parameters, learns from history
4. **Finalize Agent**: Compiles results and outputs best parameters

**Implementation Status:**
- Core agentic MRO code is available at: `apps/mobility_robustness_optimization/agentic_mro/`
- Notebook integration: **Coming Soon**

---

In [28]:
# TODO: Integrate agentic MRO optimization here